# Uniform random policy evaluation

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
"""One fresh RANDOM development episode using the frozen matched interface."""
from pathlib import Path
import argparse,json,os,sys,time,traceback,signal
ROOT=TRACE_ROOT
from contract import CODE,read_json,write_json,sha
def main():
    p=argparse.ArgumentParser();p.add_argument('--resolved',required=True);p.add_argument('--stage-dir',required=True)
    p.add_argument('--simulator-seed',type=int,required=True);p.add_argument('--action-seed',type=int,required=True);a=p.parse_args()
    directory=Path(a.stage_dir);directory.mkdir(exist_ok=False);cfg=read_json(a.resolved);started=time.perf_counter();runtime=None
    os.environ['MPLCONFIGDIR']=str(directory/'config-home/matplotlib');os.environ['XDG_CONFIG_HOME']=str(directory/'config-home')
    os.chdir(CODE);sys.path.insert(0,str(CODE))
    def interrupt(signum,frame):raise KeyboardInterrupt(f'signal {signum}')
    signal.signal(signal.SIGTERM,interrupt);signal.signal(signal.SIGINT,interrupt)
    result={'status':'RUNNING','mode':'evaluate','condition':'RANDOM','pid':os.getpid(),'config_sha256':sha(a.resolved),
        'simulator_seed':a.simulator_seed,'action_sampler_seed':a.action_seed,'PPO_learning_decisions':0,'checkpoint_sha256':None}
    print('RESOLVED_SETTINGS '+json.dumps(cfg,sort_keys=True),flush=True)
    try:
        import numpy as np
        import torch
        torch.set_num_threads(1);torch.set_num_interop_threads(1)
        from runtime import Runtime
        write_json(directory/'runtime_versions.json',{'python':sys.version,'numpy':np.__version__,'torch':torch.__version__,'device':'cpu'})
        runtime_cfg=dict(cfg);runtime_cfg['condition']=cfg['adapter_reward_condition']
        runtime=Runtime(runtime_cfg,directory,a.simulator_seed);env=runtime.create()
        rng=np.random.default_rng(a.action_seed);obs,info=env.reset();loop_start=time.perf_counter();samples=[]
        result['setup_seconds_before_loop']=loop_start-started
        for count in range(1,601):
            action=rng.integers(0,3,size=2,dtype=np.int64)
            obs,reward,terminated,truncated,info=env.step(action)
            if count%100==0:
                def mem(pid):
                    return {line.split(':')[0]:int(line.split()[1])*1024 for line in Path(f'/proc/{pid}/status').read_text().splitlines() if line.startswith(('VmRSS:','VmHWM:'))}
                samples.append({'decisions':count,'python':mem(os.getpid()),'unity':mem(runtime.process.pid)})
            if terminated or truncated:break
        else:raise AssertionError('No ending by horizon')
        result.update(status='PASS',evaluation_decisions=count,loop_seconds=time.perf_counter()-loop_start,
            raw_score=info['raw_score_signal'],terminated=terminated,truncated=truncated,episode_end_reason=info['episode_end_reason'])
        write_json(directory/'resource_samples.json',samples)
    except BaseException as exc:result.update(status='FAIL',exception=repr(exc));traceback.print_exc()
    finally:
        if runtime is not None:
            try:result['runtime']=runtime.close()
            except BaseException as exc:result.update(status='FAIL',cleanup_error=repr(exc));traceback.print_exc()
        result['wall_seconds']=time.perf_counter()-started;write_json(directory/'result.json',result)
        print('STAGE_RESULT '+json.dumps(result),flush=True)
    return 0 if result['status']=='PASS' else 1
print('Uniform random policy evaluation definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Uniform random policy evaluation definitions/execution completed.
